# Week 6 · Notebook 1 — Build a Tiny Neural Network
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

We train a small 2-layer classifier on a simple **two-moons** dataset and watch the loss go down. This is the exact machinery — neurons, layers, activations, loss, gradient descent, backprop — that scales up to LLMs.

> **Runs in Colab.** If PyTorch will not import we fall back to a from-scratch **NumPy** network so you still see the forward/backward loop.

## 0. Setup
PyTorch is preinstalled in Colab. The import is guarded so this notebook still runs (NumPy fallback) if torch is unavailable.

In [ ]:
import numpy as np
try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
    print('PyTorch', torch.__version__, '— using PyTorch path')
except Exception as e:
    HAS_TORCH = False
    print('PyTorch not available -> NumPy fallback. (', e, ')')

## 1. A simple dataset (two interleaving moons)
Two classes that a straight line *cannot* separate — so we need a hidden layer with a nonlinear activation. We make the data with plain NumPy so there are no extra dependencies.

In [ ]:
rng = np.random.default_rng(0)

def make_moons(n=400, noise=0.20):
    n2 = n // 2
    t = np.linspace(0, np.pi, n2)
    # outer moon
    x0 = np.c_[np.cos(t), np.sin(t)]
    # inner moon, shifted
    x1 = np.c_[1 - np.cos(t), 0.5 - np.sin(t)]
    X = np.vstack([x0, x1]) + noise * rng.standard_normal((2 * n2, 2))
    y = np.r_[np.zeros(n2), np.ones(n2)].astype(int)
    idx = rng.permutation(len(y))
    return X[idx].astype('float32'), y[idx]

X, y = make_moons()
print('X', X.shape, 'y', y.shape, 'classes', np.unique(y))

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=12)
plt.title('Two moons — not linearly separable'); plt.show()

## 2a. Train with PyTorch
A `Sequential` model = a hidden `Linear` layer + `ReLU`, then a 2-class output. `CrossEntropyLoss` applies softmax internally. The loop is the universal recipe: **forward → loss → backward (backprop) → step**.

In [ ]:
def train_torch(X, y, hidden=16, lr=0.05, epochs=200):
    Xt = torch.tensor(X)
    yt = torch.tensor(y, dtype=torch.long)
    model = nn.Sequential(
        nn.Linear(2, hidden), nn.ReLU(),
        nn.Linear(hidden, 2),
    )
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        opt.zero_grad()
        logits = model(Xt)         # forward
        loss = loss_fn(logits, yt)
        loss.backward()            # backprop (autograd)
        opt.step()                 # gradient-descent update
        history.append(loss.item())
    acc = (model(Xt).argmax(1) == yt).float().mean().item()
    return model, history, acc

if HAS_TORCH:
    model, history, acc = train_torch(X, y)
    print(f'final loss={history[-1]:.4f}  train accuracy={acc:.2%}')

## 2b. NumPy fallback — the same network, by hand
If torch is missing, we implement one hidden layer + ReLU + softmax and **code the backward pass ourselves** so you can see every gradient. Same five ideas, no framework.

In [ ]:
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def train_numpy(X, y, hidden=16, lr=0.1, epochs=400):
    n, d = X.shape; K = 2
    W1 = 0.5 * rng.standard_normal((d, hidden)); b1 = np.zeros(hidden)
    W2 = 0.5 * rng.standard_normal((hidden, K)); b2 = np.zeros(K)
    Y = np.eye(K)[y]                      # one-hot targets
    history = []
    for epoch in range(epochs):
        # ---- forward ----
        Z1 = X @ W1 + b1; A1 = np.maximum(0, Z1)   # ReLU
        Z2 = A1 @ W2 + b2; P = softmax(Z2)
        loss = -np.mean(np.sum(Y * np.log(P + 1e-9), axis=1))
        history.append(loss)
        # ---- backward (chain rule) ----
        dZ2 = (P - Y) / n
        dW2 = A1.T @ dZ2; db2 = dZ2.sum(0)
        dA1 = dZ2 @ W2.T; dZ1 = dA1 * (Z1 > 0)     # ReLU grad
        dW1 = X.T @ dZ1; db1 = dZ1.sum(0)
        # ---- gradient-descent update ----
        W1 -= lr * dW1; b1 -= lr * db1
        W2 -= lr * dW2; b2 -= lr * db2
    acc = (softmax((np.maximum(0, X @ W1 + b1)) @ W2 + b2).argmax(1) == y).mean()
    return (W1, b1, W2, b2), history, acc

if not HAS_TORCH:
    params, history, acc = train_numpy(X, y)
    print(f'final loss={history[-1]:.4f}  train accuracy={acc:.2%}')

## 3. Watch the loss go down
Whichever path ran, `history` holds the loss at every epoch. A falling curve means gradient descent is working.

In [ ]:
plt.plot(history)
plt.xlabel('epoch'); plt.ylabel('loss')
plt.title('Training loss — gradient descent at work'); plt.show()

## 4. Experiment (do this for the midterm prep)
Rerun training while changing one thing at a time and note the effect:
1. **Learning rate** — try 0.001, 0.05, 0.5. What breaks?
2. **Hidden size** — try 2, 8, 64. Does accuracy change?
3. **Epochs** — too few = underfit; very many = does it help?

Write 3–4 sentences on what you observed.

In [ ]:
# Your experiments here
